# FreightIQ — Model 1: Raw Data Inspection

## Objective
Inspect all raw freight-market and economic datasets before preprocessing and integration.

## Purpose
This step verifies dataset structure, date coverage, data types, missing values, and duplicates.
No cleaning or transformation is performed at this stage.

In [8]:
# ============================================================
# FREIGHTIQ — COMPLETE RAW DATA INSPECTION
# Model 1: Freight Market Forecasting
# ============================================================

from pathlib import Path
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Define raw data folder
# ------------------------------------------------------------

RAW_DIR = Path("../data/raw")

FREIGHT_DIR = RAW_DIR / "freight_indices"
MARKET_DIR = RAW_DIR / "market"

print("=" * 80)
print("FREIGHTIQ — RAW DATA INSPECTION")
print("=" * 80)


# ============================================================
# 2. Helper function for CSV inspection
# ============================================================

def inspect_csv(file_path, name):
    print("\n" + "=" * 80)
    print(f"{name}")
    print("=" * 80)

    df = pd.read_csv(file_path)

    print(f"File       : {file_path.name}")
    print(f"Rows       : {len(df):,}")
    print(f"Columns    : {len(df.columns)}")
    print(f"Column names:")
    print(list(df.columns))

    print("\nFirst 5 rows:")
    display(df.head())

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    # Check Date column
    if "Date" in df.columns:
        dates = pd.to_datetime(df["Date"], errors="coerce")

        print("\nDate information:")
        print(f"Valid dates : {dates.notna().sum():,}")
        print(f"Invalid dates: {dates.isna().sum():,}")

        if dates.notna().any():
            print(f"Start date  : {dates.min().date()}")
            print(f"End date    : {dates.max().date()}")

        print(f"Duplicate dates: {dates.duplicated().sum():,}")

    return df


# ============================================================
# FREIGHT INDEX FILES — CORRECT PATHS
# ============================================================

freight_files = {
    "BDI — 1985 to 2000":
        FREIGHT_DIR / "Baltic Dry Index" / "BDI_1985-2000.csv",

    "BDI — 2001 to 2016":
        FREIGHT_DIR / "Baltic Dry Index" / "BDI_2001-2016.csv",

    "BDI — 2017 to 2026":
        FREIGHT_DIR / "Baltic Dry Index" / "BDI_2017-2026.csv",

    "BCI — 2012 to 2017":
        FREIGHT_DIR / "Baltic Capesize" / "2012-2017.csv",

    "BCI — 2016 to 2025":
        FREIGHT_DIR / "Baltic Capesize" / "2016-2025.csv",

    "BPI — 2012 to 2025":
        FREIGHT_DIR / "Baltic Panamax" / "2012-2025.csv",

    "BSI — 2012 to 2025":
        FREIGHT_DIR / "Baltic Supramax" / "2012-25.csv",

    "BHSI — 2012 to 2025":
        FREIGHT_DIR / "Baltic Handysize" / "2012-2025.csv",
}

freight_data = {}

for name, file_path in freight_files.items():

    if file_path.exists():
        freight_data[name] = inspect_csv(file_path, name)
    else:
        print(f"WARNING: File not found → {file_path}")
freight_data = {}

for name, file_path in freight_files.items():

    if file_path.exists():
        freight_data[name] = inspect_csv(file_path, name)

    else:
        print(f"\nWARNING: File not found → {file_path}")


# ============================================================
# 4. Brent Crude Oil
# ============================================================

brent_file = MARKET_DIR / "Brent Crude Oil Price 1987-2026.csv"

if brent_file.exists():

    print("\n" + "=" * 80)
    print("BRENT CRUDE OIL")
    print("=" * 80)

    brent = pd.read_csv(brent_file)

    print(f"File       : {brent_file.name}")
    print(f"Rows       : {len(brent):,}")
    print(f"Columns    : {len(brent.columns)}")
    print(f"Columns    : {list(brent.columns)}")

    print("\nFirst 5 rows:")
    display(brent.head())

    print("\nData types:")
    print(brent.dtypes)

    print("\nMissing values:")
    print(brent.isna().sum())

    # Normalize date column
    if "observation_date" in brent.columns:

        brent["Date"] = pd.to_datetime(
            brent["observation_date"],
            errors="coerce"
        )

        print("\nDate information:")
        print(f"Valid dates  : {brent['Date'].notna().sum():,}")
        print(f"Invalid dates: {brent['Date'].isna().sum():,}")
        print(f"Start date   : {brent['Date'].min().date()}")
        print(f"End date     : {brent['Date'].max().date()}")
        print(f"Duplicate dates: {brent['Date'].duplicated().sum():,}")

    freight_data["Brent"] = brent


# ============================================================
# 5. USD / INR — RBI HTML disguised as XLS
# ============================================================

usd_file = MARKET_DIR / "USD_INR Exchange Rate.xls"

print("\n" + "=" * 80)
print("USD / INR EXCHANGE RATE")
print("=" * 80)

print(f"File exists : {usd_file.exists()}")

if usd_file.exists():

    print(f"File size   : {usd_file.stat().st_size:,} bytes")

    # IMPORTANT:
    # RBI file is HTML content saved with .xls extension.
    print("\nReading RBI HTML table...")

    try:

        usd_tables = pd.read_html(usd_file)

        print(f"Tables found: {len(usd_tables)}")

        for i, table in enumerate(usd_tables):

            print("\n" + "-" * 70)
            print(f"TABLE {i}")
            print("-" * 70)

            print(f"Rows    : {len(table):,}")
            print(f"Columns : {len(table.columns)}")
            print(f"Columns : {list(table.columns)}")

            display(table.head(10))

    except Exception as e:

        print("\nERROR while reading USD/INR file:")
        print(e)


# ============================================================
# 6. World Bank Commodity Prices
# ============================================================

commodity_file = MARKET_DIR / "Monthly Prices Coal + Iron Ore.xlsx"

print("\n" + "=" * 80)
print("WORLD BANK COMMODITY DATA")
print("=" * 80)

if commodity_file.exists():

    print(f"File : {commodity_file.name}")

    try:

        excel_file = pd.ExcelFile(commodity_file)

        print("\nAvailable sheets:")
        print(excel_file.sheet_names)

        # Read the correct sheet
        commodity = pd.read_excel(
            commodity_file,
            sheet_name="Monthly Prices",
            header=None
        )

        print("\nMonthly Prices sheet:")
        print(f"Rows    : {len(commodity):,}")
        print(f"Columns : {len(commodity.columns)}")

        print("\nFirst 8 rows:")
        display(commodity.head(8))

        print("\nImportant commodity columns:")

        # Row 4 contains commodity names
        commodity_headers = commodity.iloc[4]

        for i, value in commodity_headers.items():

            if pd.notna(value):

                text = str(value).lower()

                if "coal" in text or "iron ore" in text:

                    print(f"Column {i}: {value}")

    except Exception as e:

        print("\nERROR while reading commodity file:")
        print(e)

else:

    print(f"WARNING: File not found → {commodity_file}")


# ============================================================
# 7. Final Inspection Summary
# ============================================================

print("\n" + "=" * 80)
print("RAW DATA INSPECTION COMPLETED")
print("=" * 80)

print("""
Next step:
1. Review the USD/INR table output.
2. Confirm the correct RBI table.
3. Extract and clean USD/INR.
4. Then we will calculate the common date range
   across all Model 1 variables.
""")

FREIGHTIQ — RAW DATA INSPECTION

BDI — 1985 to 2000
File       : BDI_1985-2000.csv
Rows       : 3,794
Columns    : 7
Column names:
['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']

First 5 rows:


,Date,Price,Open,High,Low,Vol.,Change %
0,01/04/1985,"1,000.00","1,000.00","1,000.00","1,000.00",NaN,63.93%
1,01/07/1985,998.50,998.50,998.50,998.50,NaN,-0.15%
2,01/08/1985,996.50,996.50,996.50,996.50,NaN,-0.20%
3,01/09/1985,994.50,994.50,994.50,994.50,NaN,-0.20%
4,01/10/1985,979.50,979.50,979.50,979.50,NaN,-1.51%



Data types:
Date            str
Price           str
Open            str
High            str
Low             str
Vol.        float64
Change %        str
dtype: object

Missing values:
Date           0
Price          0
Open           0
High           0
Low            0
Vol.        3794
Change %       0
dtype: int64

Date information:
Valid dates : 3,794
Invalid dates: 0
Start date  : 1985-01-04
End date    : 2000-01-31
Duplicate dates: 0

BDI — 2001 to 2016
File       : BDI_2001-2016.csv
Rows       : 3,999
Columns    : 7
Column names:
['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']

First 5 rows:


,Date,Price,Open,High,Low,Vol.,Change %
0,12/23/2016,961.00,961.00,961.00,961.00,NaN,3.56%
1,12/22/2016,928.00,928.00,928.00,928.00,NaN,0.22%
2,12/21/2016,926.00,926.00,926.00,926.00,NaN,1.31%
3,12/20/2016,914.00,914.00,914.00,914.00,NaN,-1.40%
4,12/19/2016,927.00,926.00,927.00,926.00,NaN,-2.01%



Data types:
Date            str
Price           str
Open            str
High            str
Low             str
Vol.        float64
Change %        str
dtype: object

Missing values:
Date           0
Price          0
Open           0
High           0
Low            0
Vol.        3999
Change %       0
dtype: int64

Date information:
Valid dates : 3,999
Invalid dates: 0
Start date  : 2001-01-02
End date    : 2016-12-23
Duplicate dates: 0

BDI — 2017 to 2026
File       : BDI_2017-2026.csv
Rows       : 2,422
Columns    : 7
Column names:
['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']

First 5 rows:


...


Data types:
Date            str
Price           str
Open            str
High            str
Low             str
Vol.        float64
Change %        str
dtype: object

Missing values:
Date           0
Price          0
Open           0
High           0
Low            0
Vol.        3173
Change %       0
dtype: int64

Date information:
Valid dates : 3,173
Invalid dates: 0
Start date  : 2012-07-04
End date    : 2025-03-31
Duplicate dates: 0

BRENT CRUDE OIL
File       : Brent Crude Oil Price 1987-2026.csv
Rows       : 10,270
Columns    : 2
Columns    : ['observation_date', 'DCOILBRENTEU']

First 5 rows:


,observation_date,DCOILBRENTEU
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60
4,1987-05-26,18.63



Data types:
observation_date        str
DCOILBRENTEU        float64
dtype: object

Missing values:
observation_date       0
DCOILBRENTEU        1173
dtype: int64

Date information:
Valid dates  : 10,270
Invalid dates: 0
Start date   : 1987-05-20
End date     : 2026-09-29
Duplicate dates: 0

USD / INR EXCHANGE RATE
File exists : True
File size   : 135,676 bytes

Reading RBI HTML table...

ERROR while reading USD/INR file:
`Import lxml` failed.  Use pip or conda to install the lxml package.

WORLD BANK COMMODITY DATA
File : Monthly Prices Coal + Iron Ore.xlsx

Available sheets:
['Mismatch Details', 'Monthly Prices', 'Monthly Indices', 'Description', 'Index Weights']

Monthly Prices sheet:
Rows    : 807
Columns : 72

First 8 rows:


,0,1,2,3,4,5,6,7,8,9,...,62,63,64,65,66,67,68,69,70,71
0,World Bank Commodity Price Data (The Pink Sheet),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"monthly prices in nominal US dollars, 1960 to ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(monthly series are available only in nominal ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Updated on October 02, 2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African **","Natural gas, US","Natural gas, Europe","Liquefied natural gas, Japan",...,Aluminum,"Iron ore, cfr spot",Copper,Lead,Tin,Nickel,Zinc,Gold,Platinum,Silver
5,NaN,($/bbl),($/bbl),($/bbl),($/bbl),($/mt),($/mt),($/mmbtu),($/mmbtu),($/mmbtu),...,($/mt),($/dmtu),($/mt),($/mt),($/mt),($/mt),($/mt),($/troy oz),($/troy oz),($/troy oz)
6,1960M01,1.6,1.6,1.6,…,…,…,0.14,0.4,…,...,511,11.4,715,206,2180,1631,261,35,84,0.9
7,1960M02,1.6,1.6,1.6,…,…,…,0.14,0.4,…,...,511,11.4,728,204,2180,1631,245,35,84,0.9



Important commodity columns:
Column 5: Coal, Australian
Column 6: Coal, South African **
Column 63: Iron ore, cfr spot

RAW DATA INSPECTION COMPLETED

Next step:
1. Review the USD/INR table output.
2. Confirm the correct RBI table.
3. Extract and clean USD/INR.
4. Then we will calculate the common date range
   across all Model 1 variables.



In [13]:
import pandas as pd
from pathlib import Path

usd_file = Path("../data/raw/market/USD_INR Exchange Rate.xls")

print("=" * 80)
print("USD / INR — RBI DATA INSPECTION")
print("=" * 80)

tables = pd.read_html(usd_file)

print(f"Tables found: {len(tables)}")

for i, table in enumerate(tables):

    print("\n" + "-" * 70)
    print(f"TABLE {i}")
    print("-" * 70)

    print(f"Rows    : {len(table):,}")
    print(f"Columns : {len(table.columns)}")
    print(f"Columns : {list(table.columns)}")

    display(table.head(10))

USD / INR — RBI DATA INSPECTION
Tables found: 1

----------------------------------------------------------------------
TABLE 0
----------------------------------------------------------------------
Rows    : 2,657
Columns : 2
Columns : [0, 1]


,0,1
0,Date,USD (INR / 1 USD)
1,01/10/2026,95.9927
2,30/09/2026,95.9832
3,29/09/2026,96.0321
4,28/09/2026,95.9681
5,25/09/2026,95.8918
6,24/09/2026,95.9099
7,23/09/2026,95.7310
8,22/09/2026,95.8179
9,21/09/2026,95.7991


In [14]:
# ============================================================
# USD / INR — CLEANING & VALIDATION
# ============================================================

import pandas as pd
from pathlib import Path

usd_file = Path("../data/raw/market/USD_INR Exchange Rate.xls")

# Read RBI HTML table
usd_raw = pd.read_html(usd_file)[0]

# First row contains the actual column names
usd_raw.columns = usd_raw.iloc[0]

# Remove the header row
usd = usd_raw.iloc[1:].copy()

# Rename columns
usd.columns = ["Date", "USD_INR"]

# Convert data types
usd["Date"] = pd.to_datetime(
    usd["Date"],
    format="%d/%m/%Y",
    errors="coerce"
)

usd["USD_INR"] = pd.to_numeric(
    usd["USD_INR"],
    errors="coerce"
)

# Sort chronologically
usd = usd.sort_values("Date").reset_index(drop=True)

# ============================================================
# VALIDATION
# ============================================================

print("=" * 80)
print("USD / INR — CLEANED DATA")
print("=" * 80)

print(f"Rows              : {len(usd):,}")
print(f"Columns           : {len(usd.columns)}")
print(f"Missing dates     : {usd['Date'].isna().sum():,}")
print(f"Missing USD/INR   : {usd['USD_INR'].isna().sum():,}")
print(f"Duplicate dates   : {usd['Date'].duplicated().sum():,}")

print(f"\nStart date        : {usd['Date'].min().date()}")
print(f"End date          : {usd['Date'].max().date()}")

print("\nData types:")
print(usd.dtypes)

print("\nFirst 5 rows:")
display(usd.head())

print("\nLast 5 rows:")
display(usd.tail())

print("\nDescriptive statistics:")
display(usd["USD_INR"].describe())

USD / INR — CLEANED DATA
Rows              : 2,656
Columns           : 2
Missing dates     : 0
Missing USD/INR   : 0
Duplicate dates   : 0

Start date        : 2012-01-02
End date          : 2026-10-01

Data types:
Date       datetime64[us]
USD_INR           float64
dtype: object

First 5 rows:


,Date,USD_INR
0,2012-01-02,53.2975
1,2012-01-03,53.2288
2,2012-01-04,53.1400
3,2012-01-05,52.7810
4,2012-01-06,52.7838



Last 5 rows:


,Date,USD_INR
2651,2026-09-25,95.8918
2652,2026-09-28,95.9681
2653,2026-09-29,96.0321
2654,2026-09-30,95.9832
2655,2026-10-01,95.9927



Descriptive statistics:


count    2656.000000
mean       71.472860
std        12.488823
min        48.679000
25%        62.156900
50%        66.847700
75%        83.183025
max        96.844100
Name: USD_INR, dtype: float64

In [17]:
# ============================================================
# FREIGHTIQ — CORRECT COMMON DATE RANGE
# ============================================================

# Combined coverage of each logical dataset

logical_ranges = {
    "BDI": {
        "Start": pd.Timestamp("1985-01-04"),
        "End": pd.Timestamp("2026-10-02")
    },

    "BCI": {
        "Start": pd.Timestamp("2012-07-04"),
        "End": pd.Timestamp("2025-03-31")
    },

    "BPI": {
        "Start": pd.Timestamp("2012-07-04"),
        "End": pd.Timestamp("2025-03-31")
    },

    "BSI": {
        "Start": pd.Timestamp("2012-07-04"),
        "End": pd.Timestamp("2025-03-27")
    },

    "BHSI": {
        "Start": pd.Timestamp("2012-07-04"),
        "End": pd.Timestamp("2025-03-31")
    },

    "Brent": {
        "Start": pd.Timestamp("1987-05-20"),
        "End": pd.Timestamp("2026-09-29")
    },

    "USD_INR": {
        "Start": usd["Date"].min(),
        "End": usd["Date"].max()
    }
}

logical_range_df = pd.DataFrame(logical_ranges).T

print("=" * 80)
print("MODEL 1 — LOGICAL DATASET DATE RANGES")
print("=" * 80)

display(logical_range_df)

# Correct common range
common_start = logical_range_df["Start"].max()
common_end = logical_range_df["End"].min()

print("\n" + "=" * 80)
print("FINAL COMMON DATE RANGE")
print("=" * 80)

print(f"Start : {common_start.date()}")
print(f"End   : {common_end.date()}")
print(f"Days  : {(common_end - common_start).days:,}")

MODEL 1 — LOGICAL DATASET DATE RANGES


,Start,End
BDI,1985-01-04,2026-10-02
BCI,2012-07-04,2025-03-31
BPI,2012-07-04,2025-03-31
BSI,2012-07-04,2025-03-27
BHSI,2012-07-04,2025-03-31
Brent,1987-05-20,2026-09-29
USD_INR,2012-01-02,2026-10-01



FINAL COMMON DATE RANGE
Start : 2012-07-04
End   : 2025-03-27
Days  : 4,649
